<a href="https://colab.research.google.com/github/annaphuongwit/14_LLMs/blob/main/5_chunking_strategy_KE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
project_root_dir = "/Users/karimelbana/wbs/rag_project_DSAI/"
sys.path.append(project_root_dir)

# Part 5: Evaluating Chunking Strategies

This guide provides instructions for the next phase of optimising your RAG application: systematically testing different chunking strategies. Having established a baseline performance, we will now experiment with how we split our documents to see if we can improve retrieval quality.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

---
## 1.&nbsp; The Importance of Chunking

The way we break down our documents into smaller pieces, or chunks, is one of the most critical factors in the performance of a RAG system.

* If chunks are too small, they may not contain enough context for the LLM to understand the information properly.
* If chunks are too large, they might contain too much irrelevant information, creating noise that confuses the retriever.

The goal is to find the "Goldilocks zone" for chunk size and overlap that provides the retriever with just the right amount of context.

It’s also important to remember a key technical constraint: your chosen chunk_size must be smaller than the maximum number of tokens your embedding model can handle. Most modern embedding models have a large context window, but it’s a good practice to be aware of this limit.


![Screenshot 2025-09-22 at 20.31.53.png](<attachment:Screenshot 2025-09-22 at 20.31.53.png>)

In [ ]:
# pooling all tokens into 1 vector

---
## 2.&nbsp; Preparing for the Experiment

To test different chunking strategies, we'll add a new evaluation stage to our existing framework. This involves adding a new configuration list and a new function to our evaluation scripts.

### 2.1 Add the Chunking Strategy Configurations

First, we need to define the different chunk sizes and overlaps we want to test. We will centralise this in our evaluation configuration file.

1.  Open `evaluation/evaluation_config.py` in your VSCode editor.
2.  Add the following list to the end of the file. This list defines the experiments we will run.

In [ ]:
# --- Configuration for Chunking Strategy Evaluation ---
CHUNKING_STRATEGY_CONFIGS: list[dict[str, int]] = [ # = param_grid in GridSearchCV or RandomizedSearchCV
    {'size': 512, 'overlap': 50},
    {'size': 768, 'overlap': 115},
    {'size': 1024, 'overlap': 200},
]
CHUNKING_STRATEGY_CONFIGS # analogy: similar to batch_size

# the larger the chunck-size the more context the less precision
# the larger the overlap the more context

[{'size': 512, 'overlap': 50},
 {'size': 768, 'overlap': 115},
 {'size': 1024, 'overlap': 200}]

In [ ]:
# a book 100 pages every page has 500 words (50,000 words whole book: max llm window-size>):

# assuming 0 overlap

# for chunk size 500 words:
# -> you get 100 chunks -> each chunk will be a 700 long vector
# -> we retirev the most similar 4 vectors to our prompt
# -> we retrieved 4 * 500 words = 2000 words

# for chunk size 1000 words:
# -> you get 50 chunks -> each chunk will be a 700 long vector
# -> we retirev the most similar 4 vectors to our prompt
# -> we retrieved 4 * 1000 words = 4000 words

This setup makes it easy to add more experimental configurations in the future without altering the evaluation logic.

### 2.2 Making the Index Builder More Robust

Before we write the new evaluation loop, we need to make a small but important change to the `get_or_build_index` function in `evaluation/run_evaluation.py`. Currently, it saves every experimental index with the same name. We need to give each index a unique name based on its parameters. This ensures that if the script is interrupted, it can resume where it left off without rebuilding completed vector stores.

1.  Open `evaluation/run_evaluation.py`.
2.  Find the `get_or_build_index` function.
3.  Change the line that defines `vector_store_id` to the following:

In [ ]:
# Change this line:
vector_store_id: str = f"vs_baseline"

# To this:
vector_store_id: str = f"vs_chunk_{chunk_size}_overlap_{chunk_overlap}"

NameError: name 'chunk_size' is not defined

### 2.3 Create the Chunking Evaluation Function

Now, we'll add a new function to `evaluation/run_evaluation.py` that is responsible for testing our chunking strategies.

1.  Open `evaluation/run_evaluation.py` in your VSCode editor.
2.  First, import the new configuration list by adding `CHUNKING_STRATEGY_CONFIGS` to the existing import from `evaluation.evaluation_config`.

In [ ]:
from evaluation.eval_config import (
    EVAL_METRICS,
    EVALUATION_RESULTS_PATH,
    EXPERIMENTAL_VECTOR_STORES_PATH,
    CHUNKING_STRATEGY_CONFIGS, # Add this line
)

/Users/karimelbana/.pyenv/versions/3.10.6/envs/rag_dsai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


3. Now, add the following function to the file, placing it after the `evaluate_baseline` function.

In [ ]:
def evaluate_chunking_strategies(
    llm: Groq,
    embed_model: HuggingFaceEmbedding
) -> pd.DataFrame:
    """ Evaluates different chunk sizes and overlaps. """
    print("\n--- 🚀 Stage 2: Evaluating Chunking Strategies ---")
    questions, ground_truths = get_eval_data()
    all_results: list[pd.DataFrame] = []

    for config in CHUNKING_STRATEGY_CONFIGS:
        chunk_size, chunk_overlap = config['size'], config['overlap']
        print(f"\n--- Testing Chunk Config: size={chunk_size}, overlap={chunk_overlap} ---")

        index: VectorStoreIndex = get_or_build_index(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap, embed_model=embed_model
        )
        query_engine: BaseQueryEngine = index.as_query_engine(similarity_top_k=SIMILARITY_TOP_K, llm=llm)

        qa_dataset: Dataset = generate_qa_dataset(query_engine, questions, ground_truths)

        print("--- Running Ragas evaluation for chunking... ---")

        result: Dataset = evaluate(
            dataset=qa_dataset,
            metrics=EVAL_METRICS,
            raise_exceptions=True,
        )

        results_df: pd.DataFrame = result.to_pandas()
        results_df['chunk_size'] = chunk_size
        results_df['chunk_overlap'] = chunk_overlap
        all_results.append(results_df)

    final_df: pd.DataFrame = pd.concat(all_results, ignore_index=True)
    save_results(final_df, "chunking_evaluation")
    print("--- ✅ Chunking Strategy Evaluation Complete ---")
    return final_df

This function iterates through each configuration in `CHUNKING_STRATEGY_CONFIGS`, builds a unique vector store for that configuration, runs the Ragas evaluation, and then combines all the results into a single DataFrame.

### 2.4 Update the Main Execution Block

Finally, we need to call our new function.

In `evaluation/run_evaluation.py`, update the `if __name__ == "__main__":` block at the end of the file to include a call to `evaluate_chunking_strategies`.

In [ ]:
if __name__ == "__main__":
    llm_to_test: Groq = initialise_llm()
    embed_model_to_test: HuggingFaceEmbedding = get_embedding_model()

    # Run Stage 1: Baseline Evaluation
    # evaluate_baseline(llm=llm_to_test, embed_model=embed_model_to_test)

    # Run Stage 2: Chunking Strategy Evaluation
    evaluate_chunking_strategies(llm=llm_to_test, embed_model=embed_model_to_test)

---
## 3.&nbsp; Run the Chunking Evaluation

With the new function and configurations in place, you're ready to run the experiment.

From your VSCode terminal, run the evaluation script as before:

In [ ]:
python -m evaluation.run_evaluation

The script will first run the baseline evaluation and then proceed to Stage 2, testing each chunking strategy in turn. It will create and save a separate vector store for each strategy in the `local_storage/experimental_vector_stores` directory.

---
## 4.&nbsp; Analyse the Results

Once the script finishes, navigate to the `evaluation/evaluation_results` folder. You will find two new files:

1.  **`chunking_evaluation_detailed_... .csv`**: Contains the row-by-row scores for every question under each chunking strategy.
2.  **`chunking_evaluation_summary_... .csv`**: This is the most important file. It shows the average scores for each metric, grouped by `chunk_size` and `chunk_overlap`.

Open the summary file. Your goal is to find the combination of `chunk_size` and `chunk_overlap` that yields the best overall scores, particularly for **`context_recall`** and **`context_precision`**, as these are most directly affected by the quality of the retrieved chunks.

---
## 5.&nbsp; Challenges and Further Exploration 😀

You've now established a repeatable process for optimising a crucial component of your RAG pipeline. The next step is to apply these techniques to the custom RAG chatbot you have been building with your own data.

### 1. Find Your Optimal Strategy for Your Project

Now it's time to run the chunking evaluation on your own RAG system and analyse the results to find the best settings for your specific documents.

**Your Mission:**

1.  **Run the Full Evaluation:**

2.  **Set Up Your Analysis Notebook:**

      * In your `notebooks` directory, create a new Notebook file named `02_chunking_analysis.ipynb`.

3.  **Load and Analyse Your Results:**

      * In the new notebook, load the summary results from your chunking experiment.
      * Analyse the trade-offs. Which combination of `chunk_size` and `chunk_overlap` gives the best `context_recall` and `context_precision`? Create plots to help visualise the differences.

4.  **Document Your Decision and Update Your Application:**

      * In a markdown cell, write a conclusion explaining which chunking strategy you have chosen as the new optimum for your project and justify your decision using the data.
      * Open `src/config.py` and update the `CHUNK_SIZE` and `CHUNK_OVERLAP` variables with your new, optimised values. Your main chatbot will now use this improved configuration.

### 2. Expand the Experiment

The three strategies we tested are just a starting point. Now that you have a robust evaluation pipeline, you can easily conduct more advanced experiments.

**Challenge:**

  * Return to `evaluation/evaluation_config.py` and add more configurations to the `CHUNKING_STRATEGY_CONFIGS` list. Try a wider range of sizes and overlaps.
  * Does a very large chunk size (e.g., 2048) improve performance by providing more context, or does it introduce too much noise and lower the scores?
  * What happens with a very small chunk size (e.g., 256)? Does it improve precision at the cost of recall?
  * Run the evaluation again and analyse the new results to see if you can improve performance further.

In [ ]:
import pandas as pd

results = pd.read_csv('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/evaluation_results/chunking_evaluation_summary_2025-09-23_21-16-24.csv')
results

,chunk_size,chunk_overlap,faithfulness,context_precision,context_recall
0,512,50,0.750000,0.500000,0.700000
1,768,115,0.440278,0.555556,0.357143
2,1024,200,0.568056,0.555556,0.500000


In [ ]:
import numpy as np

results['mean_score'] = results.apply(lambda row: np.mean([
    row['faithfulness'], row['context_precision'], row['context_recall']]
    ),axis = 1)
results

,chunk_size,chunk_overlap,faithfulness,context_precision,context_recall,mean_score
0,512,50,0.750000,0.500000,0.700000,0.650000
1,768,115,0.440278,0.555556,0.357143,0.450992
2,1024,200,0.568056,0.555556,0.500000,0.541204


In [ ]:
best_chunk_size = int(results.sort_values(by='mean_score', ascending=False).head(1)['chunk_size'].values[0])
best_chunk_size

512

In [ ]:
best_chunk_overlap = int(results.sort_values(by='mean_score', ascending=False).head(1)['chunk_overlap'].values[0])
best_chunk_overlap

50